In [ ]:
import pandas as pd
import os


class FeatureEngineering:

    def __init__(self, filepath):

        self.filepath = filepath

        self.df = None

        os.makedirs("../outputs/features", exist_ok=True)

    ########################################################

    def load_data(self):

        print("=" * 60)
        print("Loading Clean Dataset")
        print("=" * 60)

        self.df = pd.read_csv(
            self.filepath,
            parse_dates=["InvoiceDate"],
            dtype={
                "Invoice": str,
                "StockCode": str
            },
            low_memory=False
        )

        print(self.df.shape)

    ########################################################

    def create_time_features(self):

        print("\nCreating Time Features...")

        self.df["Year"] = self.df["InvoiceDate"].dt.year

        self.df["Quarter"] = self.df["InvoiceDate"].dt.quarter

        self.df["Month"] = self.df["InvoiceDate"].dt.month

        self.df["Month_Name"] = self.df["InvoiceDate"].dt.month_name()

        self.df["Week"] = self.df["InvoiceDate"].dt.isocalendar().week.astype(int)

        self.df["Day"] = self.df["InvoiceDate"].dt.day

        self.df["Weekday"] = self.df["InvoiceDate"].dt.day_name()

        self.df["Hour"] = self.df["InvoiceDate"].dt.hour

        self.df["Weekend"] = self.df["Weekday"].isin(
            ["Saturday", "Sunday"]
        ).astype(int)

    ########################################################

    def customer_features(self):

        print("Creating Customer Features...")

        customer = (

            self.df

            .groupby("Customer_ID")

            .agg(

                Total_Revenue=("Total_Price", "sum"),

                Total_Orders=("Invoice", "nunique"),

                Total_Items=("Quantity", "sum"),

                Avg_Order_Value=("Total_Price", "mean"),

                Unique_Products=("StockCode", "nunique"),

                Last_Purchase=("InvoiceDate", "max")

            )

            .reset_index()

        )

        latest = self.df["InvoiceDate"].max()

        customer["Recency"] = (

            latest - customer["Last_Purchase"]

        ).dt.days

        customer.to_csv(

            "../outputs/features/customer_features.csv",

            index=False

        )

        print(customer.shape)

    ########################################################

    def product_features(self):

        print("Creating Product Features...")

        product = (

            self.df

            .groupby("StockCode")

            .agg(

                Description=("Description", "first"),

                Revenue=("Total_Price", "sum"),

                Quantity=("Quantity", "sum"),

                Orders=("Invoice", "nunique"),

                Customers=("Customer_ID", "nunique"),

                Avg_Price=("Price", "mean")

            )

            .reset_index()

        )

        product.to_csv(

            "../outputs/features/product_features.csv",

            index=False

        )

        print(product.shape)

    ########################################################

    def country_features(self):

        print("Creating Country Features...")

        country = (

            self.df

            .groupby("Country")

            .agg(

                Revenue=("Total_Price", "sum"),

                Customers=("Customer_ID", "nunique"),

                Orders=("Invoice", "nunique"),

                Quantity=("Quantity", "sum")

            )

            .reset_index()

        )

        country.to_csv(

            "../outputs/features/country_features.csv",

            index=False

        )

        print(country.shape)

    ########################################################

    def save_master_dataset(self):

        self.df.to_csv(

            "../outputs/features/master_feature_dataset.csv",

            index=False

        )

        print("Master Dataset Saved")

    ########################################################

    def run(self):

        self.load_data()

        self.create_time_features()

        self.customer_features()

        self.product_features()

        self.country_features()

        self.save_master_dataset()

        print("\nFeature Engineering Completed Successfully")


if __name__ == "__main__":

    obj = FeatureEngineering(

        r"C:\Users\ka843\Coding\Amdox Internship_project\data\clean\clean_sales.csv"

    )

    obj.run()